# Model-CDR

In the modeling stage, I am going to follow the instructions to build three different classification models to determine whether or not a CT scan should be recommended.

In this section, I will focus on

**The clinical decision rule from Kuppermann et al.**


In [1]:
import numpy as np
import pandas as pd

# import the clean and preprocess functions
from preprocess import preprocess_data
# import constants
from features import numeric_cols, binary_cols, multi_category_cols, parent_child_groups, categorical_cols

In [2]:
pd.set_option('display.max_columns', None)
pd.options.display.max_colwidth = 500
pd.options.display.max_rows = 100

## Data Preprocessing

1. Data filtering
   
   In the cleaning stage, I don't remove the patients with GCS 3-13. To completely follow the steps introduced in the paper, I will do the exclusion.
   As for the train-val-test split, I have done the splitting **stratified by age and final result**, which is aligned with data flow chat in the paper

2. Categorical Features preparation
    As for two models, we have 6 predictors for each models, some variables should be prepared base on the existing raw features.

    Prediction Tree for children younger than 2 years
      + Altered mental status (0: No, 1: Yes)
      + scalp haematoma (Occipital or parietal or temporal)
      + loss of consciousness (>=5 s)
      + Mechanism of injury (severe)
      + Palpable or unclear skull fracture 
      + Acting normally per parent? (No)
    
    Any predictor present should be suggested to get a CT scan

    Prediction Tree for chidren aged 2 years and older
     + Altered mental status
     + Loss of consciousness (Yes or suspected)
     + History of vomiting
     + Mechanism of injury (severe)
     + Clinical signs of basilar skull fracture
     + Severe headache



In [3]:
%run clean.py

In [4]:
%who

List	 binary_cols	 categorical_cols	 clean_data	 flag_posintfinal_inconsistency	 multi_category_cols	 np	 numeric_cols	 parent_child_groups	 
pd	 preprocess_data	 tbi_cleaned_new	 tbi_origin	 


In [5]:
tbi_cleaned_new = pd.read_csv('../data/TBI_cleaned.csv')
tbi_cleaned_new.shape

(43379, 126)

In [6]:
train_cdr_data, val_cdr_data, test_cdr_data = preprocess_data(
    df=tbi_cleaned_new,
    numeric_cols=numeric_cols,
    categorical_cols=categorical_cols,
    multi_category_cols=multi_category_cols,
    binary_cols=binary_cols,
    parent_child_groups=parent_child_groups,
    target_col='PosIntFinal',
    test_size=0.2,
    val_size=0.1,
    random_state=42,
    stratify_col=['PosIntFinal', 'AgeTwoPlus'],
    if_exclude_gcs_under_13= True, # drop GCS 3-13 as paper
    if_fill_missing_gcs=True,
    gcs_fill_strategy="leave", # leave missing 
    if_one_hot_encode=False,
    drop_first_cat_in_ohe=True,
    if_drop_na_rows=False,
    if_handle_parent_child_missing=True,
    if_handle_missing_for_categories=True,
    missing_category_label="missing",
    if_add_flag_missing_cols=False,
    parent_missing_strategy="leave",
    child_missing_when_parent_yes="missing_category",
    binary_missing_strategy="leave",
    multi_missing_strategy="missing_category"
)

In [7]:
print("Train shape:", train_cdr_data.shape)
print("Val shape:", val_cdr_data.shape)
print("Test shape:", test_cdr_data.shape)

Train shape: (29687, 126)
Val shape: (4242, 126)
Test shape: (8483, 126)


The number of observations is aligned with the paper's data description.

## Modeling



**Prediction Tree for children younger than 2 years**
  + Altered mental status
  + scalp haematoma (Occipital or parietal or temporal)
  + loss of consciousness (>=5 s)
  + Mechanism of injury (severe)
  + Palpable or unclear skull fracture 
  + Acting normally per parent? (No)

Any predictor present should be suggested to get a CT scan

**Prediction Tree for chidren aged 2 years and older**
  + Altered mental status
  + Loss of consciousness (Yes or suspected)
  + History of vomiting
  + Mechanism of injury (severe)
  + Clinical signs of basilar skull fracture
  + Severe headache

Any predictor present should be suggested to get a CT scan

In [8]:
from typing import Any, Dict, Iterable, Optional, Tuple, Union

def _is_missing(x: Any) -> bool:
    if pd.isna(x):
        return True
    if isinstance(x, str) and x.strip() == '':
        return True
    return False

In [9]:
from dataclasses import dataclass
@dataclass
class PECARNColumns:
    """
    Default column mapping for your dataset (adjust if your names differ).

    Age group:
      - AgeTwoPlus: 1 => <2 years, 2 => >=2 years  (per your preprocessing)
    <2 years rule variables:
      - AMS
      - HemaLoc: location of scalp hematoma (needs to detect occipital/parietal/temporal vs none/frontal)
      - LOCSeparate + LocLen: LOC indicator and duration (seconds); rule uses >=5 seconds
      - High_impact_InjSev: severe mechanism indicator (1/0) or a coded severity you can threshold
      - SFxPalp: palpable skull fracture
      - ActNorm: acting normally per parent (1 yes / 0 no)
    >=2 years rule variables:
      - AMS
      - LOCSeparate (any LOC)
      - Vomit (history of vomiting)
      - High_impact_InjSev (severe mechanism)
      - SFxBas (basilar skull fracture signs)
      - HASeverity (severe headache)  (you can map to severe if HASeverity==1 etc)
    """

    AgeTwoPlus: str = "AgeTwoPlus"

    # shared
    AMS: str = "AMS"
    High_impact_InjSev: str = "High_impact_InjSev"

    # <2
    HemaLoc: str = "HemaLoc"
    LocLen: str = "LocLen"
    SFxPalp: str = "SFxPalp"
    ActNorm: str = "ActNorm"

    # >=2
    Vomit: str = "Vomit"
    SFxBas: str = "SFxBas"
    HASeverity: str = "HASeverity"
    LOCSeparate: str = "LOCSeparate"

In [10]:
class PECARNDecisionRule:
    """
    Replicates the PECARN 'very low risk' rules (Kuppermann et al., 2009):
      - Separate rules for age < 2 and age >= 2.
      - Output: 0 => very low risk (no predictors present)
                1 => not very low risk (>=1 predictor present OR unknown required field)

    IMPORTANT (clinical / modeling-safe):
      - Any unknown/missing on a required decision node is treated conservatively as "not very low risk".
    """

    def __init__(
        self,
        cols: Optional[PECARNColumns] = None,
        *,
        loc_severe_codes: Optional[Iterable[Any]] = {1,2},
        loc_len_ge_5s_code: {Any} = {2, 3, 4},
        severe_mech_codes: Optional[Iterable[Any]] = {3},
        severe_headache_codes: Optional[Iterable[Any]] = {3},
        hema_opt_codes: Optional[Iterable[Any]] = {2,3},
        palpalable_sfx_codes: Optional[Iterable[Any]] = {1,2},
    ):
        self.cols = cols or PECARNColumns()
        self.loc_len_ge_5s_code = loc_len_ge_5s_code
        self.loc_severe_codes = set(loc_severe_codes) if loc_severe_codes is not None else set()
        self.severe_mech_codes = set(severe_mech_codes) if severe_mech_codes is not None else set()
        self.severe_headache_codes = (
            set(severe_headache_codes) if severe_headache_codes is not None else set()
        )
        self.hema_opt_codes = set(hema_opt_codes) if hema_opt_codes is not None else set()
        self.palpalable_sfx_codes = set(palpalable_sfx_codes) if palpalable_sfx_codes is not None else set()    

    # --- helpers to interpret coded fields ---
    def _is_ams(self, x: Any) -> Optional[int]:
        if _is_missing(x):
            return None
        return int(x == 1)
    def _is_vomit(self, x: Any) -> Optional[int]:
        if _is_missing(x):
            return None
        return int(x == 1)
    def _is_bas_skull(self, x: Any) -> Optional[int]:
        if _is_missing(x):
            return None
        return int(x == 1)

    def _is_severe_mechanism(self, x: Any) -> Optional[int]:
        if _is_missing(x):
            return None
        if self.severe_mech_codes is None:
            # no mapping provided and not 0/1 -> unknown
            return None
        return int(x in self.severe_mech_codes)

    def _is_severe_headache(self, x: Any) -> Optional[int]:
        if _is_missing(x):
            return None
        if self.severe_headache_codes is None:
            return None
        return int(x in self.severe_headache_codes)

    def _hema_is_opt(self, x: Any) -> Optional[int]:
        if _is_missing(x):
            return None
        if self.hema_opt_codes is None:
            return None
        return int(x in self.hema_opt_codes)

    def _loc_ge_5s(self, loc_sep: Any, loc_len: Any) -> Optional[int]:
        if _is_missing(loc_len ):
            return None
        if self.loc_len_ge_5s_code is None:
            return None
        return int(loc_len in self.loc_len_ge_5s_code)
    
    def _loc_severe(self, loc_sep: Any) -> Optional[int]:
        if _is_missing(loc_sep):
            return None
        if self.loc_severe_codes is None:
            return None
        return int(loc_sep in self.loc_severe_codes)
    
    def _sfx_palp_severe(self, x: Any) -> Optional[int]:
        if _is_missing(x):
            return None
        if self.palpalable_sfx_codes is None:
            return None
        return int(x in self.palpalable_sfx_codes)
    
    def _act_norm_not_normal(self, x: Any) -> Optional[int]:
        if _is_missing(x):
            return None
        return int(x == 0)

    
    # --- core rules ---

    def _predict_row(self, row: pd.Series) -> Dict[str, Any]:
        c = self.cols

        agegrp = row.get(c.AgeTwoPlus, None)

        # Normalize age group to bool: <2 => 1, >=2 => 2 (your convention)
        is_under2 = (agegrp == 1) 

        reasons = []

        # Shared: AMS is a predictor for both trees
        ams = self._is_ams(row.get(c.AMS))
        if ams is None:
            reasons.append("AMS missing/unknown")
        elif ams == 1:
            reasons.append("Altered mental status = Yes")

        severe_mech = self._is_severe_mechanism(row.get(c.High_impact_InjSev))
        if severe_mech is None:
            # in the tree it's used later, but missing still prevents 'rule out' if reached
            pass

        if is_under2:
            # <2 years predictors (any one => not very low risk)
            hema_opt = self._hema_is_opt(row.get(c.HemaLoc))
            loc_ge5 = self._loc_ge_5s(row.get(c.LOCSeparate), row.get(c.LocLen))
            sfx_palp = self._sfx_palp_severe(row.get(c.SFxPalp))
            act_norm = self._act_norm_not_normal(row.get(c.ActNorm))

            # The derivation tree structure is sequential, but the published "no predictor present"
            # summary corresponds to: none of these predictors are present.
            # Missing in any predictor => cannot be confidently "no predictor present".
            predictors = {
                "AMS": ams,
                "Scalp hematoma (O/P/T)": hema_opt,
                "LOC >= 5s": loc_ge5,
                "Severe mechanism": severe_mech,
                "Palpable skull fracture": sfx_palp,
                "Not acting normally per parent": act_norm,
            }

            for name, val in predictors.items():
                if val is None:
                    reasons.append(f"{name} missing/unknown")
                elif val == 1:
                    reasons.append(f"{name} present")

            # very low risk only if ALL predictors are known and ==0
            if all(v == 0 for v in predictors.values() if v is not None):
                return {"very_low_risk": 1, "rule": "<2", "reasons": ["No PECARN predictors present"]}
            return {"very_low_risk": 0, "rule": "<2", "reasons": reasons}

        else:
            # >=2 years predictors
            loc_any = self._loc_severe(row.get(c.LOCSeparate))
            vomit = self._is_vomit(row.get(c.Vomit))
            sfx_bas = self._is_bas_skull(row.get(c.SFxBas))
            ha_severe = self._is_severe_headache(row.get(c.HASeverity))

            predictors = {
                "AMS": ams,
                "Any LOC": loc_any,
                "History of vomiting": vomit,
                "Severe mechanism": severe_mech,
                "Basilar skull fracture signs": sfx_bas,
                "Severe headache": ha_severe,
            }

            for name, val in predictors.items():
                if val is None:
                    reasons.append(f"{name} missing/unknown")
                elif val == 1:
                    reasons.append(f"{name} present")

            if all(v == 0 for v in predictors.values() if v is not None):
                return {"very_low_risk": 1, "rule": ">=2", "reasons": ["No PECARN predictors present"]}
            return {"very_low_risk": 0, "rule": ">=2", "reasons": reasons}

    # --- public API ---

    def predict(self, X: Union[pd.DataFrame, Dict[str, Any]]) -> np.ndarray:
        """
        Returns array y_hat where:
          0 => very low risk (no predictors present)   [safe to avoid CT under PECARN context]
          1 => not very low risk
        """
        if isinstance(X, dict):
            row = pd.Series(X)
            out = self._predict_row(row)
            return np.array([0 if out["very_low_risk"] == 1 else 1], dtype=int)

        preds = []
        for _, row in X.iterrows():
            out = self._predict_row(row)
            preds.append(0 if out["very_low_risk"] == 1 else 1)
        return np.array(preds, dtype=int)

    def predict_very_low_risk(self, X: Union[pd.DataFrame, Dict[str, Any]]) -> np.ndarray:
        """1 => very low risk, 0 => not very low risk."""
        if isinstance(X, dict):
            out = self._predict_row(pd.Series(X))
            return np.array([out["very_low_risk"]], dtype=int)
        return np.array([self._predict_row(r)["very_low_risk"] for _, r in X.iterrows()], dtype=int)

    def explain(self, x: Union[pd.Series, Dict[str, Any]]) -> Dict[str, Any]:
        """Return rule used and reasons (useful for reporting/debugging)."""
        row = x if isinstance(x, pd.Series) else pd.Series(x)
        return self._predict_row(row)



In [11]:

pecarn = PECARNDecisionRule(
    cols=PECARNColumns(
        AgeTwoPlus="AgeTwoPlus",
        AMS="AMS",
        High_impact_InjSev="High_impact_InjSev",
        HemaLoc="HemaLoc",
        LOCSeparate="LOCSeparate",
        LocLen="LocLen",
        SFxPalp="SFxPalp",
        ActNorm="ActNorm",
        Vomit="Vomit",
        SFxBas="SFxBas",
        HASeverity="HASeverity",
    )
)

y_hat = pecarn.predict(train_cdr_data) 
train_cdr_data["PECARN_pred"] = y_hat

/var/folders/4l/rnbg3z_9549618f5h72zn7040000gn/T/ipykernel_26502/3949445853.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_cdr_data["PECARN_pred"] = y_hat


Qucik check the logics in some cases

In [12]:
# idx: 40987, age < 2
train_cdr_data.loc[40987, ["AgeTwoPlus",'InjuryMech', "AMS", "HemaLoc", "LOCSeparate", "LocLen", "High_impact_InjSev",
                            "SFxPalp", "ActNorm", "Vomit", "SFxBas", "HASeverity", "PECARN_pred",'PosIntFinal','CTDone']]

AgeTwoPlus               1
InjuryMech             8.0
AMS                    0.0
HemaLoc                1.0
LOCSeparate            0.0
LocLen                92.0
High_impact_InjSev     3.0
SFxPalp                0.0
ActNorm                0.0
Vomit                  0.0
SFxBas                 0.0
HASeverity            92.0
PECARN_pred              1
PosIntFinal            0.0
CTDone                   1
Name: 40987, dtype: object

sample 40987 should assigned as high risk due to severe injury of falling from elevation and it did get a CT scan in practice

In [13]:
# idx: 31929, age >= 2
train_cdr_data.loc[31929, ["AgeTwoPlus",'InjuryMech', "AMS", "HemaLoc", "LOCSeparate", "LocLen", "High_impact_InjSev",
                            "SFxPalp", "ActNorm", "Vomit", "SFxBas", "HASeverity", "PECARN_pred",'PosIntFinal','CTDone']]

AgeTwoPlus                  2
InjuryMech                6.0
AMS                       1.0
HemaLoc                   1.0
LOCSeparate               1.0
LocLen                missing
High_impact_InjSev        1.0
SFxPalp                   0.0
ActNorm                   NaN
Vomit                     1.0
SFxBas                    0.0
HASeverity                2.0
PECARN_pred                 1
PosIntFinal               1.0
CTDone                      1
Name: 31929, dtype: object

sample 31929 should be assigned as high risk due to AMS although the severity of injury mechanism is low, and it did get a CT scan in reality and has ciTBI indeed.

In [14]:
# since we don't exactly train the cdr model, we will not use the validation sets for hyperparameter tuning

val_cdr_data["PECARN_pred"] = pecarn.predict(val_cdr_data)
test_cdr_data["PECARN_pred"] = pecarn.predict(test_cdr_data)

/var/folders/4l/rnbg3z_9549618f5h72zn7040000gn/T/ipykernel_26502/2629466792.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  val_cdr_data["PECARN_pred"] = pecarn.predict(val_cdr_data)
/var/folders/4l/rnbg3z_9549618f5h72zn7040000gn/T/ipykernel_26502/2629466792.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_cdr_data["PECARN_pred"] = pecarn.predict(test_cdr_data)


## Evaluation

I will focus on the metrics of **Negative Predictive Value** and **Sensitivity** mainly used in the paper.

$$\text{Negative Predictive Value} = \frac{\text{\# No predicator present and no ciTBI}}{\text{\# No predicator present}}$$

$$\text{Sensitivity} = \frac{\text{\# Any predictor present and ciTBI}}{\text{\# ciTBI}}$$

In [15]:
# helper function to print out the distribution of PECARN predictions and outcomes
def print_pecarn_distribution(df: pd.DataFrame, dataset_name: str):
    print(f"--- {dataset_name} ---")
    print("PECARN prediction distribution:")
    print(df["PECARN_pred"].value_counts(normalize=True))
    print("\nContingency table with PosIntFinal:")
    print(pd.crosstab(df["PECARN_pred"], df["PosIntFinal"], rownames=["PECARN Prediction"], colnames=["ciTBI"]))
    print('Negative predictive value: {:.2f}'.format(
        pd.crosstab(df["PECARN_pred"], df["PosIntFinal"])[0][0] / (pd.crosstab(df["PECARN_pred"], df["PosIntFinal"]).to_numpy()[0,:].sum()) *100
    ))
    print('Sensitivity: {:.2f}'.format(
        pd.crosstab(df["PECARN_pred"], df["PosIntFinal"])[1][1] / (pd.crosstab(df["PECARN_pred"], df["PosIntFinal"]).to_numpy()[:,1].sum()) *100
    ))
    print("Specificity: {:.2f}".format(
        pd.crosstab(df["PECARN_pred"], df["PosIntFinal"])[0][0] / (pd.crosstab(df["PECARN_pred"], df["PosIntFinal"]).to_numpy()[:,0].sum()) *100
    ))


In [16]:
# age <2 
print_pecarn_distribution(train_cdr_data.query('AgeTwoPlus == 1'), "Train")

--- Train ---
PECARN prediction distribution:
PECARN_pred
0    0.532987
1    0.467013
Name: proportion, dtype: float64

Contingency table with PosIntFinal:
ciTBI               0.0  1.0
PECARN Prediction           
0                  3998    1
1                  3436   68
Negative predictive value: 99.97
Sensitivity: 98.55
Specificity: 53.78


In the child younger than 2 group, the prediction rules have the sensitivity of 98.55\% (68/69) and negative predictive value of 99.97\% (3998/3999)

In [17]:
print_pecarn_distribution(val_cdr_data.query('AgeTwoPlus == 1'), "Validation")

--- Validation ---
PECARN prediction distribution:
PECARN_pred
0    0.539179
1    0.460821
Name: proportion, dtype: float64

Contingency table with PosIntFinal:
ciTBI              0.0  1.0
PECARN Prediction          
0                  578    0
1                  484   10
Negative predictive value: 100.00
Sensitivity: 100.00
Specificity: 54.43


In [18]:
print_pecarn_distribution(test_cdr_data.query('AgeTwoPlus == 1'), "Test")

--- Test ---
PECARN prediction distribution:
PECARN_pred
0    0.537098
1    0.462902
Name: proportion, dtype: float64

Contingency table with PosIntFinal:
ciTBI               0.0  1.0
PECARN Prediction           
0                  1151    0
1                   973   19
Negative predictive value: 100.00
Sensitivity: 100.00
Specificity: 54.19


in the validation and test group, the sensitivity is 100\%, 100\% (10/10, 19/19) respectively, while the negative predictive value is 100\%, 100\% (578/578, 1151/1151)

In [19]:
# age >=2
print_pecarn_distribution(train_cdr_data.query('AgeTwoPlus == 2'), "Train")

--- Train ---
PECARN prediction distribution:
PECARN_pred
0    0.570636
1    0.429364
Name: proportion, dtype: float64

Contingency table with PosIntFinal:
ciTBI                0.0  1.0
PECARN Prediction            
0                  12653    6
1                   9337  188
Negative predictive value: 99.95
Sensitivity: 96.91
Specificity: 57.54


In group aged 2 years and older, the prediction rule shows sensitivity of 96.91\% and negative predictive value of 99.95\%

In [20]:
print_pecarn_distribution(val_cdr_data.query('AgeTwoPlus == 2'), "Validation")
print_pecarn_distribution(test_cdr_data.query('AgeTwoPlus == 2'), "Test")

--- Validation ---
PECARN prediction distribution:
PECARN_pred
0    0.576656
1    0.423344
Name: proportion, dtype: float64

Contingency table with PosIntFinal:
ciTBI               0.0  1.0
PECARN Prediction           
0                  1827    1
1                  1315   27
Negative predictive value: 99.95
Sensitivity: 96.43
Specificity: 58.15
--- Test ---
PECARN prediction distribution:
PECARN_pred
0    0.57776
1    0.42224
Name: proportion, dtype: float64

Contingency table with PosIntFinal:
ciTBI               0.0  1.0
PECARN Prediction           
0                  3661    2
1                  2623   54
Negative predictive value: 99.95
Sensitivity: 96.43
Specificity: 58.26


Again, in the validation group, the sensitivity and negative predictive value are 96.43\%, 99.95\% respectively.

In the test group, the sensitivity and negative predictive value are 99.95\%, 96.43\%  respectively.

#### How many CT scans can be obviated due to predicted low risk of ciTBI?

Age < 2

In [21]:
# helper function to print prediction and exact CT assignment 
def print_prediction_and_ct_decision(df: pd.DataFrame, age_group: int):
    df['PosIntFinal'] = pd.to_numeric(df['PosIntFinal'], errors='coerce')
    df['CTDone'] = pd.to_numeric(df['CTDone'], errors='coerce')

    out = df.query(f"AgeTwoPlus == {age_group}").groupby("PECARN_pred")\
        .agg({'PatNum': 'count',"PosIntFinal": 'sum','CTDone': 'sum'}).reset_index()\
        .rename(columns={"PatNum": "Count", "PosIntFinal": "ciTBI count", "CTDone": "CT count"})
    print(out)
    print('Ratio of CT scans can be avoided under PECARN rule:{:.2f}%'.format(
        out.loc[out["PECARN_pred"] == 0, "CT count"].values[0]/ out['CT count'].sum() *100
    ))

In [22]:
print_prediction_and_ct_decision(train_cdr_data, age_group=1)

   PECARN_pred  Count  ciTBI count  CT count
0            0   3999          1.0       584
1            1   3504         68.0      1743


Ratio of CT scans can be avoided under PECARN rule:25.10%


In [23]:
print('======Validation set results======')
print_prediction_and_ct_decision(val_cdr_data, age_group=1)
print('======Test set results======')
print_prediction_and_ct_decision(test_cdr_data, age_group=1)

======Validation set results======
   PECARN_pred  Count  ciTBI count  CT count
0            0    578          0.0        84
1            1    494         10.0       249
Ratio of CT scans can be avoided under PECARN rule:25.23%
======Test set results======
   PECARN_pred  Count  ciTBI count  CT count
0            0   1151          0.0       182
1            1    992         19.0       484
Ratio of CT scans can be avoided under PECARN rule:27.33%


Based on the above analysis, applying the prediction rule can reduce the CT usage in children younger than 2 by 25.10\%, 25.23\% and 27.33\% in train, validation and test datasets respectively.

Age >= 2

In [24]:
print('======Training set results======')
print_prediction_and_ct_decision(train_cdr_data, age_group=2)
print('======Validation set results======')
print_prediction_and_ct_decision(val_cdr_data, age_group=2)
print('======Test set results======')
print_prediction_and_ct_decision(test_cdr_data, age_group=2)

======Training set results======
   PECARN_pred  Count  ciTBI count  CT count
0            0  12659          6.0      1651
1            1   9525        188.0      6477
Ratio of CT scans can be avoided under PECARN rule:20.31%
======Validation set results======
   PECARN_pred  Count  ciTBI count  CT count
0            0   1828          1.0       253
1            1   1342         27.0       919
Ratio of CT scans can be avoided under PECARN rule:21.59%
======Test set results======
   PECARN_pred  Count  ciTBI count  CT count
0            0   3663          2.0       494
1            1   2677         54.0      1849
Ratio of CT scans can be avoided under PECARN rule:21.08%


Similarly, in those aged 2 and older, the application of prediction rule can result in 20.31\%, 21.59\% and 21.08\% reduction in the CT usage in train, validation and test datasets respectively.

## Interpretability

The yes or no rule is the simplest and most straightforward way for decision.

## Stability

Because the clinical decision rule is fixed (not trained), we assessed stability via a perturbation analysis that mimics plausible charting noise in manually collected clinical variables. 

Specifically, several subjective binary predictors used in the rule (e.g., altered mental status, vomiting, basilar skull fracture signs, and acting normally per parent) were randomly perturbed. In each simulation, 5% of observed binary values were flipped (0↔1), mimicking realistic variability arising from documentation differences or inter-observer assessment. Missing values were left unchanged to preserve the original data structure. This perturbation procedure was repeated 100 times on the validation dataset, and performance metrics were recomputed after each perturbation.

In [25]:
def perturb_binary(series, flip_prob=0.05, rng=None):
    rng = rng or np.random.default_rng()
    s = series.copy()
    mask = rng.random(len(s)) < flip_prob
    # only flip known 0/1; leave missing as-is
    flip_idx = mask & s.isin([0, 1])
    s.loc[flip_idx] = 1 - s.loc[flip_idx]
    return s

def run_rule_perturbation_stability(df, rule, target="PosIntFinal",
                                   flip_prob=0.05, B=50, seed=42):
    rng = np.random.default_rng(seed)
    metrics = []

    base = df.copy()

    for b in range(B):
        d = base.copy()

        # Example: perturb a few subjective binary fields 
        for col in ["AMS", "Vomit", "SFxBas", "ActNorm"]:  
            if col in d.columns:
                d[col] = perturb_binary(d[col], flip_prob=flip_prob, rng=rng)

        y_true = d[target].astype(int).values
        y_pred = rule.predict(d)   # 1=not low risk, 0=very low risk (or your convention)

        # compute simple metrics
        tp = ((y_pred == 1) & (y_true == 1)).sum()
        fn = ((y_pred == 0) & (y_true == 1)).sum()
        tn = ((y_pred == 0) & (y_true == 0)).sum()
        fp = ((y_pred == 1) & (y_true == 0)).sum()

        sens = tp / (tp + fn) if (tp + fn) else np.nan
        spec = tn / (tn + fp) if (tn + fp) else np.nan
        npv = tn / (tn + fn) if (tn + fn) else np.nan

        metrics.append({"rep": b, "sensitivity": sens, "specificity": spec, "npv": npv})

    return pd.DataFrame(metrics)

In [26]:
stab = run_rule_perturbation_stability(val_cdr_data, pecarn, flip_prob=0.05, B=100)

stab.agg(["mean", "std", "min", "max"])

,rep,sensitivity,specificity,npv
mean,49.500000,0.972895,0.502954,0.999513
std,29.011492,0.012094,0.003802,0.000217
min,0.000000,0.921053,0.493340,0.998568
max,99.000000,1.000000,0.512369,1.000000


In [27]:
stab.quantile([0.025, 0.5, 0.975])

,rep,sensitivity,specificity,npv
0.025,2.475,0.947368,0.496069,0.999048
0.500,49.500,0.973684,0.503092,0.999527
0.975,96.525,1.000000,0.509277,1.000000


In [28]:
stab.head()

,rep,sensitivity,specificity,npv
0,0,0.973684,0.509277,0.999533
1,1,0.973684,0.506898,0.999531
2,2,0.973684,0.509277,0.999533
3,3,0.973684,0.496194,0.999521
4,4,0.973684,0.506422,0.999531


**Conclusion:**

Across simulations, the rule demonstrated strong stability. Sensitivity remained consistently high, with a median value of approximately 0.97 and a narrow range spanning roughly 0.95 to 1.00. Specificity showed only modest variation, centered around 0.50, while negative predictive value (NPV) remained extremely high and nearly invariant (≈0.999–1.000).

These results indicate that the clinical decision rule is robust to small perturbations in input features and that its safety-critical property — maintaining high sensitivity and NPV — is not driven by fragile or unstable measurements. Minor changes in subjective clinical inputs primarily affect specificity rather than the ability of the rule to identify patients at risk.

Overall, the perturbation analysis supports that the decision rule’s performance is structurally stable and unlikely to change substantially under reasonable variations in clinical assessment.